## PHASE A

In [2]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Add the project root (parent of notebooks/) to sys.path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("data exists:", (PROJECT_ROOT / "data").exists())

Current working directory: c:\Users\farha\code\nasa-spacecraft-telemetry\notebooks
Project root: c:\Users\farha\code\nasa-spacecraft-telemetry
src exists: True
data exists: True


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier, IsolationForest

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Phase C: Project config and helpers
from src.config import RANDOM_STATE
from src.data.loader import load_labels_metadata
from src.data.dataset_builder import (
    build_supervised_dataset,
    build_unsupervised_dataset,
)
from src.data.splitting import create_outer_channel_split

# Phase D: Notebook display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("All imports loaded successfully.")

All imports loaded successfully.


## PHASE B

In [14]:
# Phase B1: Load the anomaly metadata
# Load the benchmark metadata that contains channel IDs and anomaly intervals
# This metadata is required to rebuild the processed datasets
labels_df = load_labels_metadata()

print("Metadata shape:", labels_df.shape)
labels_df.head()

Metadata shape: (82, 5)


,chan_id,spacecraft,anomaly_sequences,class,num_values
0,P-1,SMAP,"[[2149, 2349], [4536, 4844], [3539, 3779]]","[contextual, contextual, contextual]",8505
1,S-1,SMAP,"[[5300, 5747]]",[point],7331
2,E-1,SMAP,"[[5000, 5030], [5610, 6086]]","[contextual, contextual]",8516
3,E-2,SMAP,"[[5598, 6995]]",[point],8532
4,E-3,SMAP,"[[5094, 8306]]",[point],8307


In [15]:
# Phase B2: Rebuild the supervised MSL dataset
# Rebuild the full supervised dataset for MSL channels
# This recreates the same processed feature table used in the original notebook
msl_supervised = build_supervised_dataset(
    labels_df=labels_df,
    spacecraft="MSL",
)

print("MSL supervised feature df shape:", msl_supervised["feature_df"].shape)
print("MSL channel summary shape:", msl_supervised["channel_summary_df"].shape)
print("Number of MSL channels:", msl_supervised["channel_summary_df"]["chan_id"].nunique())

MSL supervised feature df shape: (14601, 670)
MSL channel summary shape: (27, 5)
Number of MSL channels: 27


In [16]:
# Phase B3: Rebuild the supervised SMAP dataset
# Rebuild the full supervised dataset for SMAP channels
# Keeping both spacecraft datasets available makes later comparisons easier
smap_supervised = build_supervised_dataset(
    labels_df=labels_df,
    spacecraft="SMAP",
)

print("SMAP supervised feature df shape:", smap_supervised["feature_df"].shape)
print("SMAP channel summary shape:", smap_supervised["channel_summary_df"].shape)
print("Number of SMAP channels:", smap_supervised["channel_summary_df"]["chan_id"].nunique())

SMAP supervised feature df shape: (86873, 310)
SMAP channel summary shape: (54, 5)
Number of SMAP channels: 54


In [17]:
# Phase B4: Quick label summaries for both datasets
# Check the class balance in the rebuilt datasets
# This helps confirm why SMOTE is worth testing
print("MSL label counts:")
print(msl_supervised["feature_df"]["label"].value_counts().sort_index())
print()

print("SMAP label counts:")
print(smap_supervised["feature_df"]["label"].value_counts().sort_index())

MSL label counts:
label
0    12853
1     1748
Name: count, dtype: int64

SMAP label counts:
label
0    75342
1    11531
Name: count, dtype: int64


# PHASE C

In [26]:
# Phase C1: Create the MSL outer channel split and inspect its contents
# Inspect the returned split object before assuming its key names
# This helps keep the notebook aligned with the exact helper function output
msl_outer_split = create_outer_channel_split(
    channel_summary_df=msl_supervised["channel_summary_df"],
    test_size=0.2,
)

print("Type:", type(msl_outer_split))
print("Keys:", msl_outer_split.keys())

msl_outer_split

Type: <class 'dict'>
Keys: dict_keys(['dev_chan_ids', 'test_chan_ids', 'split_summary_df', 'used_stratification'])


{'dev_chan_ids': ['C-1',
  'C-2',
  'D-14',
  'D-16',
  'F-4',
  'F-5',
  'F-8',
  'M-2',
  'M-3',
  'M-4',
  'M-5',
  'M-6',
  'M-7',
  'P-11',
  'P-14',
  'P-15',
  'T-13',
  'T-4',
  'T-5',
  'T-8',
  'T-9'],
 'test_chan_ids': ['D-15', 'F-7', 'M-1', 'P-10', 'S-2', 'T-12'],
 'split_summary_df':    chan_id spacecraft  n_windows  n_anomalous_windows  anomaly_ratio outer_split
 0      C-1        MSL        447                   74       0.165548         dev
 1      C-2        MSL        405                   39       0.096296         dev
 2     D-14        MSL        520                   56       0.107692         dev
 3     D-15        MSL        426                  131       0.307512        test
 4     D-16        MSL        433                  136       0.314088         dev
 5      F-4        MSL        679                   20       0.029455         dev
 6      F-5        MSL        779                   36       0.046213         dev
 7      F-7        MSL       1005              

In [27]:
# Phase C1: Print the MSL outer split summary
# Print the final channel-level outer split used for development and held-out testing
print("Used stratification:", msl_outer_split["used_stratification"])
print("Number of dev channels:", len(msl_outer_split["dev_chan_ids"]))
print("Number of test channels:", len(msl_outer_split["test_chan_ids"]))
print("Dev channels:", msl_outer_split["dev_chan_ids"])
print("Test channels:", msl_outer_split["test_chan_ids"])

Used stratification: True
Number of dev channels: 21
Number of test channels: 6
Dev channels: ['C-1', 'C-2', 'D-14', 'D-16', 'F-4', 'F-5', 'F-8', 'M-2', 'M-3', 'M-4', 'M-5', 'M-6', 'M-7', 'P-11', 'P-14', 'P-15', 'T-13', 'T-4', 'T-5', 'T-8', 'T-9']
Test channels: ['D-15', 'F-7', 'M-1', 'P-10', 'S-2', 'T-12']


In [28]:
# Phase C2: Build the MSL development and held-out test dataframes
# Keep only rows from dev channels for model training and validation
# Keep only rows from held-out test channels for final evaluation
msl_dev_df = msl_supervised["feature_df"][
    msl_supervised["feature_df"]["chan_id"].isin(msl_outer_split["dev_chan_ids"])
].copy()

msl_test_df = msl_supervised["feature_df"][
    msl_supervised["feature_df"]["chan_id"].isin(msl_outer_split["test_chan_ids"])
].copy()

print("MSL dev feature df shape:", msl_dev_df.shape)
print("MSL test feature df shape:", msl_test_df.shape)

MSL dev feature df shape: (10664, 670)
MSL test feature df shape: (3937, 670)


In [29]:
# Phase C3: Check the label distribution after the outer split
# Check how many normal and anomalous windows are present in dev and held-out test
print("MSL dev label counts:")
print(msl_dev_df["label"].value_counts().sort_index())
print()

print("MSL test label counts:")
print(msl_test_df["label"].value_counts().sort_index())

MSL dev label counts:
label
0    9452
1    1212
Name: count, dtype: int64

MSL test label counts:
label
0    3401
1     536
Name: count, dtype: int64


In [30]:
# Phase C4: Verify that the dev and held-out test channels are fully separate
# Confirm that no channel appears in both the dev and held-out test sets
dev_channels_set = set(msl_dev_df["chan_id"].unique())
test_channels_set = set(msl_test_df["chan_id"].unique())

print("Channel overlap:", dev_channels_set.intersection(test_channels_set))
print("Dev unique channels:", len(dev_channels_set))
print("Test unique channels:", len(test_channels_set))

Channel overlap: set()
Dev unique channels: 21
Test unique channels: 6


# PHASE D

In [31]:
# Phase D1: Identify the feature columns
# Remove metadata and label-related columns so only model input features remain
non_feature_cols = [
    "chan_id",
    "split",
    "window_id",
    "start_idx",
    "end_idx",
    "label",
    "has_overlap",
    "num_overlapping_intervals",
    "overlapping_intervals",
    "spacecraft",
]

feature_cols = [col for col in msl_dev_df.columns if col not in non_feature_cols]

print("Number of feature columns:", len(feature_cols))
print("First 10 feature columns:", feature_cols[:10])

Number of feature columns: 660
First 10 feature columns: ['mean_v0', 'mean_v1', 'mean_v2', 'mean_v3', 'mean_v4', 'mean_v5', 'mean_v6', 'mean_v7', 'mean_v8', 'mean_v9']


In [32]:
# Phase D2: Build the development-set model inputs
# Convert the dev dataframe into arrays for modeling
# Keep channel IDs separately so grouped cross-validation can split by channel
X_dev = msl_dev_df[feature_cols].to_numpy()
y_dev = msl_dev_df["label"].to_numpy()
groups_dev = msl_dev_df["chan_id"].to_numpy()

print("X_dev shape:", X_dev.shape)
print("y_dev shape:", y_dev.shape)
print("groups_dev shape:", groups_dev.shape)
print("Unique dev channels:", len(np.unique(groups_dev)))

X_dev shape: (10664, 660)
y_dev shape: (10664,)
groups_dev shape: (10664,)
Unique dev channels: 21


In [33]:
# Phase D3: Build the held-out test model inputs
# Build the final held-out test arrays that will be used only for final evaluation
X_test = msl_test_df[feature_cols].to_numpy()
y_test = msl_test_df["label"].to_numpy()
groups_test = msl_test_df["chan_id"].to_numpy()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("groups_test shape:", groups_test.shape)
print("Unique test channels:", len(np.unique(groups_test)))

X_test shape: (3937, 660)
y_test shape: (3937,)
groups_test shape: (3937,)
Unique test channels: 6


In [34]:
# Phase D4: Summarize class imbalance in the development and held-out test sets
# Compute the fraction of anomalous windows to quantify class imbalance
# This helps justify testing SMOTE in the supervised models
dev_anomaly_ratio = y_dev.mean()
test_anomaly_ratio = y_test.mean()

print("Dev anomaly ratio:", round(dev_anomaly_ratio, 4))
print("Test anomaly ratio:", round(test_anomaly_ratio, 4))
print("Dev class counts:", {0: int((y_dev == 0).sum()), 1: int((y_dev == 1).sum())})
print("Test class counts:", {0: int((y_test == 0).sum()), 1: int((y_test == 1).sum())})

Dev anomaly ratio: 0.1137
Test anomaly ratio: 0.1361
Dev class counts: {0: 9452, 1: 1212}
Test class counts: {0: 3401, 1: 536}


## PHASE E

In [35]:
# Phase E1: Set up grouped cross-validation on the dev channels
# Build grouped CV folds so channels stay separated across train and validation
# This prevents leakage from overlapping windows within the same channel
gkf = GroupKFold(n_splits=4)

cv_splits = list(gkf.split(X_dev, y_dev, groups_dev))

print("Number of folds:", len(cv_splits))

for fold_id, (train_idx, val_idx) in enumerate(cv_splits):
    train_groups = np.unique(groups_dev[train_idx])
    val_groups = np.unique(groups_dev[val_idx])

    print(f"Fold {fold_id}:")
    print("  Train rows:", len(train_idx))
    print("  Val rows:", len(val_idx))
    print("  Train channels:", len(train_groups))
    print("  Val channels:", len(val_groups))
    print("  Channel overlap:", set(train_groups).intersection(set(val_groups)))

Number of folds: 4
Fold 0:
  Train rows: 7832
  Val rows: 2832
  Train channels: 16
  Val channels: 5
  Channel overlap: set()
Fold 1:
  Train rows: 8118
  Val rows: 2546
  Train channels: 16
  Val channels: 5
  Channel overlap: set()
Fold 2:
  Train rows: 7915
  Val rows: 2749
  Train channels: 15
  Val channels: 6
  Channel overlap: set()
Fold 3:
  Train rows: 8127
  Val rows: 2537
  Train channels: 16
  Val channels: 5
  Channel overlap: set()


In [36]:
# Phase E2: Helper to compute classification metrics at a chosen threshold
# Convert scores into binary predictions using the chosen threshold
# Compute both threshold-dependent and threshold-independent metrics
def compute_metrics(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_true, y_score)
    roc_auc = roc_auc_score(y_true, y_score)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "n_rows": len(y_true),
        "n_anomalous": int(y_true.sum()),
    }

In [37]:
# Phase E3: Define the threshold search grid
# Use the same threshold grid as the original notebook so comparisons stay fair
threshold_grid = np.arange(0.05, 1.00, 0.05)

print("Threshold grid:", threshold_grid)

Threshold grid: [0.05 0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65 0.7
 0.75 0.8  0.85 0.9  0.95]


In [38]:
# Phase E4: Helper to evaluate many thresholds for one set of scores
# Evaluate the same score vector across multiple thresholds
# This helps us select the threshold with the best development-set F1
def evaluate_threshold_grid(y_true, y_score, thresholds):
    rows = []
    for threshold in thresholds:
        row = compute_metrics(y_true, y_score, threshold=threshold)
        rows.append(row)
    return pd.DataFrame(rows)

# PHASE F

In [40]:
# Phase F1: Random Forest baseline grouped CV (no SMOTE)
# Train Random Forest fold by fold without SMOTE
# Store out-of-fold scores so threshold tuning uses only unbiased dev predictions
rf_oof_scores = np.zeros(len(y_dev))
rf_fold_rows = []

for fold_id, (train_idx, val_idx) in enumerate(cv_splits):
    X_train, X_val = X_dev[train_idx], X_dev[val_idx]
    y_train, y_val = y_dev[train_idx], y_dev[val_idx]
    groups_train = groups_dev[train_idx]
    groups_val = groups_dev[val_idx]

    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    rf.fit(X_train, y_train)
    y_val_score = rf.predict_proba(X_val)[:, 1]

    rf_oof_scores[val_idx] = y_val_score

    fold_metrics = compute_metrics(y_val, y_val_score, threshold=0.5)
    fold_metrics["fold_id"] = fold_id
    fold_metrics["n_train_rows"] = len(train_idx)
    fold_metrics["n_val_rows"] = len(val_idx)
    fold_metrics["n_train_channels"] = len(np.unique(groups_train))
    fold_metrics["n_val_channels"] = len(np.unique(groups_val))

    rf_fold_rows.append(fold_metrics)

rf_cv_df = pd.DataFrame(rf_fold_rows)

print("RF OOF shape:", rf_oof_scores.shape)
rf_cv_df

RF OOF shape: (10664,)


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous,fold_id,n_train_rows,n_val_rows,n_train_channels,n_val_channels
0,0.5,0.428571,0.202572,0.275109,0.304297,0.751037,63,84,2437,248,2832,311,0,7832,2832,16,5
1,0.5,0.666667,0.294314,0.408353,0.502169,0.817996,88,44,2203,211,2546,299,1,8118,2546,16,5
2,0.5,0.505263,0.514286,0.509735,0.490987,0.802126,144,141,2328,136,2749,280,2,7915,2749,15,6
3,0.5,0.650000,0.121118,0.204188,0.289255,0.752031,39,21,2194,283,2537,322,3,8127,2537,16,5


In [41]:
# Phase F2: Tune the Random Forest baseline threshold on dev OOF scores
# Use the out-of-fold dev scores to find the best operating threshold for Random Forest
rf_threshold_df = evaluate_threshold_grid(y_dev, rf_oof_scores, threshold_grid)
rf_best_row = rf_threshold_df.sort_values("f1", ascending=False).iloc[0]

print("RF best threshold row:")
print(rf_best_row.to_dict())

rf_threshold_df.sort_values("f1", ascending=False).head(10)

RF best threshold row:
{'threshold': 0.4, 'precision': 0.4530612244897959, 'recall': 0.36633663366336633, 'f1': 0.4051094890510949, 'pr_auc': 0.3681351993099491, 'roc_auc': 0.7508584279926087, 'tp': 444.0, 'fp': 536.0, 'tn': 8916.0, 'fn': 768.0, 'n_rows': 10664.0, 'n_anomalous': 1212.0}


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous
7,0.40,0.453061,0.366337,0.405109,0.368135,0.750858,444,536,8916,768,10664,1212
8,0.45,0.496259,0.328383,0.395233,0.368135,0.750858,398,404,9048,814,10664,1212
6,0.35,0.391196,0.388614,0.389901,0.368135,0.750858,471,733,8719,741,10664,1212
9,0.50,0.535256,0.275578,0.363834,0.368135,0.750858,334,290,9162,878,10664,1212
5,0.30,0.298534,0.419967,0.348989,0.368135,0.750858,509,1196,8256,703,10664,1212
10,0.55,0.569498,0.243399,0.341040,0.368135,0.750858,295,223,9229,917,10664,1212
4,0.25,0.238377,0.465347,0.315260,0.368135,0.750858,564,1802,7650,648,10664,1212
3,0.20,0.215506,0.561881,0.311528,0.368135,0.750858,681,2479,6973,531,10664,1212
2,0.15,0.197170,0.678218,0.305519,0.368135,0.750858,822,3347,6105,390,10664,1212
11,0.60,0.598997,0.197195,0.296710,0.368135,0.750858,239,160,9292,973,10664,1212


In [42]:
# Phase F3: Train final Random Forest baseline on all dev data and test on held-out MSL
# Retrain Random Forest on the full development set
# Evaluate once on the untouched held-out test set using the tuned threshold
rf_final = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_final.fit(X_dev, y_dev)
rf_test_scores = rf_final.predict_proba(X_test)[:, 1]

rf_test_metrics = compute_metrics(
    y_true=y_test,
    y_score=rf_test_scores,
    threshold=rf_best_row["threshold"],
)

print("Random Forest baseline held-out test metrics:")
print(rf_test_metrics)

Random Forest baseline held-out test metrics:
{'threshold': np.float64(0.4), 'precision': 0.2797494780793319, 'recall': 0.5, 'f1': 0.35876840696117807, 'pr_auc': 0.32857980396912256, 'roc_auc': 0.7422591906682406, 'tp': np.int64(268), 'fp': np.int64(690), 'tn': np.int64(2711), 'fn': np.int64(268), 'n_rows': 3937, 'n_anomalous': 536}


In [43]:
# Phase F4: Random Forest + SMOTE grouped CV
# Apply SMOTE only inside the current training fold
# Keep the validation fold untouched so evaluation remains leakage-safe
rf_smote_oof_scores = np.zeros(len(y_dev))
rf_smote_fold_rows = []

for fold_id, (train_idx, val_idx) in enumerate(cv_splits):
    X_train, X_val = X_dev[train_idx], X_dev[val_idx]
    y_train, y_val = y_dev[train_idx], y_dev[val_idx]
    groups_train = groups_dev[train_idx]
    groups_val = groups_dev[val_idx]

    # Apply SMOTE only to the training fold
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    rf_smote = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    rf_smote.fit(X_train_smote, y_train_smote)
    y_val_score = rf_smote.predict_proba(X_val)[:, 1]

    rf_smote_oof_scores[val_idx] = y_val_score

    fold_metrics = compute_metrics(y_val, y_val_score, threshold=0.5)
    fold_metrics["fold_id"] = fold_id
    fold_metrics["n_train_rows_before_smote"] = len(train_idx)
    fold_metrics["n_train_rows_after_smote"] = len(y_train_smote)
    fold_metrics["n_val_rows"] = len(val_idx)
    fold_metrics["n_train_channels"] = len(np.unique(groups_train))
    fold_metrics["n_val_channels"] = len(np.unique(groups_val))

    rf_smote_fold_rows.append(fold_metrics)

rf_smote_cv_df = pd.DataFrame(rf_smote_fold_rows)

print("RF + SMOTE OOF shape:", rf_smote_oof_scores.shape)
rf_smote_cv_df

RF + SMOTE OOF shape: (10664,)


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous,fold_id,n_train_rows_before_smote,n_train_rows_after_smote,n_val_rows,n_train_channels,n_val_channels
0,0.5,0.121117,0.864952,0.212480,0.354050,0.679751,269,1952,569,42,2832,311,0,7832,13862,2832,16,5
1,0.5,0.186221,0.913043,0.309348,0.429185,0.810671,273,1193,1054,26,2546,299,1,8118,14410,2546,16,5
2,0.5,0.133584,0.889286,0.232276,0.287444,0.718996,249,1615,854,31,2749,280,2,7915,13966,2749,15,6
3,0.5,0.186853,0.953416,0.312468,0.164179,0.621835,307,1336,879,15,2537,322,3,8127,14474,2537,16,5


In [44]:
# Phase F5: Tune the best threshold for Random Forest + SMOTE
# Tune the threshold for the SMOTE version using its own dev out-of-fold scores
rf_smote_threshold_df = evaluate_threshold_grid(y_dev, rf_smote_oof_scores, threshold_grid)
rf_smote_best_row = rf_smote_threshold_df.sort_values("f1", ascending=False).iloc[0]

print("RF + SMOTE best threshold row:")
print(rf_smote_best_row.to_dict())

rf_smote_threshold_df.sort_values("f1", ascending=False).head(10)

RF + SMOTE best threshold row:
{'threshold': 0.6000000000000001, 'precision': 0.20793534166054373, 'recall': 0.7004950495049505, 'f1': 0.3206798866855524, 'pr_auc': 0.24824726576658793, 'roc_auc': 0.7228944421632175, 'tp': 849.0, 'fp': 3234.0, 'tn': 6218.0, 'fn': 363.0, 'n_rows': 10664.0, 'n_anomalous': 1212.0}


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous
11,0.60,0.207935,0.700495,0.320680,0.248247,0.722894,849,3234,6218,363,10664,1212
12,0.65,0.220938,0.536304,0.312951,0.248247,0.722894,650,2292,7160,562,10664,1212
10,0.55,0.180014,0.823432,0.295441,0.248247,0.722894,998,4546,4906,214,10664,1212
9,0.50,0.152627,0.905941,0.261242,0.248247,0.722894,1098,6096,3356,114,10664,1212
13,0.70,0.214246,0.320132,0.256699,0.248247,0.722894,388,1423,8029,824,10664,1212
8,0.45,0.139324,0.952970,0.243107,0.248247,0.722894,1155,7135,2317,57,10664,1212
7,0.40,0.130055,0.971122,0.229390,0.248247,0.722894,1177,7873,1579,35,10664,1212
14,0.75,0.269704,0.180693,0.216403,0.248247,0.722894,219,593,8859,993,10664,1212
6,0.35,0.121525,0.981023,0.216260,0.248247,0.722894,1189,8595,857,23,10664,1212
5,0.30,0.118163,0.989274,0.211110,0.248247,0.722894,1199,8948,504,13,10664,1212


In [ ]:
# Phase F6: Train final Random Forest + SMOTE on all dev data and test on held-out MSL
# After threshold tuning, apply SMOTE to the full dev set only after threshold tuning is complete 
# Retrain XGBoost once and evaluate on the untouched held-out test channels
smote_final = SMOTE(random_state=RANDOM_STATE)
X_dev_smote, y_dev_smote = smote_final.fit_resample(X_dev, y_dev)

rf_smote_final = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_smote_final.fit(X_dev_smote, y_dev_smote)
rf_smote_test_scores = rf_smote_final.predict_proba(X_test)[:, 1]

rf_smote_test_metrics = compute_metrics(
    y_true=y_test,
    y_score=rf_smote_test_scores,
    threshold=rf_smote_best_row["threshold"],
)

print("Random Forest + SMOTE held-out test metrics:")
print(rf_smote_test_metrics)

Random Forest + SMOTE held-out test metrics:
{'threshold': np.float64(0.6000000000000001), 'precision': 0.206575682382134, 'recall': 0.621268656716418, 'f1': 0.3100558659217877, 'pr_auc': 0.3244396375002814, 'roc_auc': 0.7328754273326106, 'tp': np.int64(333), 'fp': np.int64(1279), 'tn': np.int64(2122), 'fn': np.int64(203), 'n_rows': 3937, 'n_anomalous': 536}


In [46]:
# Phase F7: Compare Random Forest baseline vs Random Forest + SMOTE
# Summarize the baseline and SMOTE results side by side for easy comparison
rf_compare_df = pd.DataFrame([
    {
        "model": "Random Forest baseline",
        "dev_best_threshold": rf_best_row["threshold"],
        "test_precision": rf_test_metrics["precision"],
        "test_recall": rf_test_metrics["recall"],
        "test_f1": rf_test_metrics["f1"],
        "test_pr_auc": rf_test_metrics["pr_auc"],
        "test_roc_auc": rf_test_metrics["roc_auc"],
        "test_tp": rf_test_metrics["tp"],
        "test_fp": rf_test_metrics["fp"],
        "test_tn": rf_test_metrics["tn"],
        "test_fn": rf_test_metrics["fn"],
    },
    {
        "model": "Random Forest + SMOTE",
        "dev_best_threshold": rf_smote_best_row["threshold"],
        "test_precision": rf_smote_test_metrics["precision"],
        "test_recall": rf_smote_test_metrics["recall"],
        "test_f1": rf_smote_test_metrics["f1"],
        "test_pr_auc": rf_smote_test_metrics["pr_auc"],
        "test_roc_auc": rf_smote_test_metrics["roc_auc"],
        "test_tp": rf_smote_test_metrics["tp"],
        "test_fp": rf_smote_test_metrics["fp"],
        "test_tn": rf_smote_test_metrics["tn"],
        "test_fn": rf_smote_test_metrics["fn"],
    },
])

rf_compare_df

,model,dev_best_threshold,test_precision,test_recall,test_f1,test_pr_auc,test_roc_auc,test_tp,test_fp,test_tn,test_fn
0,Random Forest baseline,0.4,0.279749,0.500000,0.358768,0.32858,0.742259,268,690,2711,268
1,Random Forest + SMOTE,0.6,0.206576,0.621269,0.310056,0.32444,0.732875,333,1279,2122,203


## PHASE G

In [ ]:
# Phase G1: XGBoost baseline grouped CV (no SMOTE)
# Train XGBoost fold by fold without SMOTE and collect out-of-fold development scores
# These scores are later used for fair threshold tuning on the dev set
xgb_oof_scores = np.zeros(len(y_dev))
xgb_fold_rows = []

for fold_id, (train_idx, val_idx) in enumerate(cv_splits):
    X_train, X_val = X_dev[train_idx], X_dev[val_idx]
    y_train, y_val = y_dev[train_idx], y_dev[val_idx]
    groups_train = groups_dev[train_idx]
    groups_val = groups_dev[val_idx]

    # Use the same imbalance-aware setting as before
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    )

    xgb.fit(X_train, y_train)
    y_val_score = xgb.predict_proba(X_val)[:, 1]

    xgb_oof_scores[val_idx] = y_val_score

    fold_metrics = compute_metrics(y_val, y_val_score, threshold=0.5)
    fold_metrics["fold_id"] = fold_id
    fold_metrics["n_train_rows"] = len(train_idx)
    fold_metrics["n_val_rows"] = len(val_idx)
    fold_metrics["n_train_channels"] = len(np.unique(groups_train))
    fold_metrics["n_val_channels"] = len(np.unique(groups_val))
    fold_metrics["scale_pos_weight"] = scale_pos_weight

    xgb_fold_rows.append(fold_metrics)

xgb_cv_df = pd.DataFrame(xgb_fold_rows)

print("XGB OOF shape:", xgb_oof_scores.shape)
xgb_cv_df

XGB OOF shape: (10664,)


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous,fold_id,n_train_rows,n_val_rows,n_train_channels,n_val_channels,scale_pos_weight
0,0.5,0.457143,0.257235,0.329218,0.337129,0.754904,80,95,2426,231,2832,311,0,7832,2832,16,5,7.692564
1,0.5,0.431159,0.397993,0.413913,0.390882,0.720300,119,157,2090,180,2546,299,1,8118,2546,16,5,7.891566
2,0.5,0.189522,0.439286,0.264801,0.214029,0.708883,123,526,1943,157,2749,280,2,7915,2749,15,6,7.492489
3,0.5,0.326241,0.142857,0.198704,0.240309,0.725592,46,95,2120,276,2537,322,3,8127,2537,16,5,8.131461


In [ ]:
# Phase G2: Tune the XGBoost baseline threshold on dev out-of-fold scores
# Search across multiple thresholds to find the operating point with the best dev-set F1
xgb_threshold_df = evaluate_threshold_grid(y_dev, xgb_oof_scores, threshold_grid)
xgb_best_row = xgb_threshold_df.sort_values("f1", ascending=False).iloc[0]

print("XGBoost best threshold row:")
print(xgb_best_row.to_dict())

xgb_threshold_df.sort_values("f1", ascending=False).head(10)

XGBoost best threshold row:
{'threshold': 0.3, 'precision': 0.24730631092868138, 'recall': 0.3976897689768977, 'f1': 0.304966782663714, 'pr_auc': 0.2769590052482419, 'roc_auc': 0.7123203010102112, 'tp': 482.0, 'fp': 1467.0, 'tn': 7985.0, 'fn': 730.0, 'n_rows': 10664.0, 'n_anomalous': 1212.0}


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous
5,0.30,0.247306,0.397690,0.304967,0.276959,0.71232,482,1467,7985,730,10664,1212
11,0.60,0.329756,0.278878,0.302190,0.276959,0.71232,338,687,8765,874,10664,1212
9,0.50,0.296535,0.303630,0.300041,0.276959,0.71232,368,873,8579,844,10664,1212
12,0.65,0.342887,0.266502,0.299907,0.276959,0.71232,323,619,8833,889,10664,1212
10,0.55,0.310559,0.288779,0.299273,0.276959,0.71232,350,777,8675,862,10664,1212
8,0.45,0.281958,0.318482,0.299109,0.276959,0.71232,386,983,8469,826,10664,1212
1,0.10,0.194472,0.644389,0.298776,0.276959,0.71232,781,3235,6217,431,10664,1212
7,0.40,0.268824,0.335809,0.298606,0.276959,0.71232,407,1107,8345,805,10664,1212
6,0.35,0.252461,0.359736,0.296700,0.276959,0.71232,436,1291,8161,776,10664,1212
2,0.15,0.202699,0.545380,0.295551,0.276959,0.71232,661,2600,6852,551,10664,1212


In [ ]:
# Phase G3: Train the final XGBoost baseline on all dev data and evaluate on held-out MSL
# Retrain once on the full dev set, then test once on the untouched held-out channels
final_scale_pos_weight = (y_dev == 0).sum() / (y_dev == 1).sum()

xgb_final = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=final_scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
)

xgb_final.fit(X_dev, y_dev)
xgb_test_scores = xgb_final.predict_proba(X_test)[:, 1]

xgb_test_metrics = compute_metrics(
    y_true=y_test,
    y_score=xgb_test_scores,
    threshold=xgb_best_row["threshold"],
)

print("XGBoost baseline held-out test metrics:")
print(xgb_test_metrics)

XGBoost baseline held-out test metrics:
{'threshold': np.float64(0.3), 'precision': 0.21049692908989392, 'recall': 0.7033582089552238, 'f1': 0.3240223463687151, 'pr_auc': 0.2529376877286542, 'roc_auc': 0.7095095494301501, 'tp': np.int64(377), 'fp': np.int64(1414), 'tn': np.int64(1987), 'fn': np.int64(159), 'n_rows': 3937, 'n_anomalous': 536}


In [ ]:
# Phase G4: XGBoost + SMOTE grouped CV
# Apply SMOTE only inside each training fold and leave validation folds untouched
# This tests whether oversampling improves XGBoost without introducing leakage
xgb_smote_oof_scores = np.zeros(len(y_dev))
xgb_smote_fold_rows = []

for fold_id, (train_idx, val_idx) in enumerate(cv_splits):
    X_train, X_val = X_dev[train_idx], X_dev[val_idx]
    y_train, y_val = y_dev[train_idx], y_dev[val_idx]
    groups_train = groups_dev[train_idx]
    groups_val = groups_dev[val_idx]

    # Apply SMOTE only to the training fold
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    # Recompute class weighting after SMOTE
    scale_pos_weight_smote = (y_train_smote == 0).sum() / (y_train_smote == 1).sum()

    xgb_smote = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_smote,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    )

    xgb_smote.fit(X_train_smote, y_train_smote)
    y_val_score = xgb_smote.predict_proba(X_val)[:, 1]

    xgb_smote_oof_scores[val_idx] = y_val_score

    fold_metrics = compute_metrics(y_val, y_val_score, threshold=0.5)
    fold_metrics["fold_id"] = fold_id
    fold_metrics["n_train_rows_before_smote"] = len(train_idx)
    fold_metrics["n_train_rows_after_smote"] = len(y_train_smote)
    fold_metrics["n_val_rows"] = len(val_idx)
    fold_metrics["n_train_channels"] = len(np.unique(groups_train))
    fold_metrics["n_val_channels"] = len(np.unique(groups_val))
    fold_metrics["scale_pos_weight_after_smote"] = scale_pos_weight_smote

    xgb_smote_fold_rows.append(fold_metrics)

xgb_smote_cv_df = pd.DataFrame(xgb_smote_fold_rows)

print("XGB + SMOTE OOF shape:", xgb_smote_oof_scores.shape)
xgb_smote_cv_df

XGB + SMOTE OOF shape: (10664,)


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous,fold_id,n_train_rows_before_smote,n_train_rows_after_smote,n_val_rows,n_train_channels,n_val_channels,scale_pos_weight_after_smote
0,0.5,0.323843,0.292605,0.307432,0.330485,0.781997,91,190,2331,220,2832,311,0,7832,13862,2832,16,5,1.0
1,0.5,0.272727,0.591973,0.373418,0.389267,0.772062,177,472,1775,122,2546,299,1,8118,14410,2546,16,5,1.0
2,0.5,0.178016,0.653571,0.279817,0.292114,0.730816,183,845,1624,97,2749,280,2,7915,13966,2749,15,6,1.0
3,0.5,0.189693,0.537267,0.280389,0.245753,0.670793,173,739,1476,149,2537,322,3,8127,14474,2537,16,5,1.0


In [ ]:
# Phase G5: Tune the best threshold for XGBoost + SMOTE
# Use the SMOTE-based out-of-fold development scores to choose the best F1 operating point
xgb_smote_threshold_df = evaluate_threshold_grid(y_dev, xgb_smote_oof_scores, threshold_grid)
xgb_smote_best_row = xgb_smote_threshold_df.sort_values("f1", ascending=False).iloc[0]

print("XGBoost + SMOTE best threshold row:")
print(xgb_smote_best_row.to_dict())

xgb_smote_threshold_df.sort_values("f1", ascending=False).head(10)

XGBoost + SMOTE best threshold row:
{'threshold': 0.4, 'precision': 0.2108029197080292, 'recall': 0.5957095709570958, 'f1': 0.3114082380849687, 'pr_auc': 0.2971300609766641, 'roc_auc': 0.7297622589173856, 'tp': 722.0, 'fp': 2703.0, 'tn': 6749.0, 'fn': 490.0, 'n_rows': 10664.0, 'n_anomalous': 1212.0}


,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,tn,fn,n_rows,n_anomalous
7,0.40,0.210803,0.595710,0.311408,0.29713,0.729762,722,2703,6749,490,10664,1212
8,0.45,0.214855,0.556106,0.309956,0.29713,0.729762,674,2463,6989,538,10664,1212
6,0.35,0.205039,0.631188,0.309529,0.29713,0.729762,765,2966,6486,447,10664,1212
5,0.30,0.199510,0.671617,0.307634,0.29713,0.729762,814,3266,6186,398,10664,1212
9,0.50,0.217422,0.514851,0.305732,0.29713,0.729762,624,2246,7206,588,10664,1212
16,0.85,0.320036,0.291254,0.304968,0.29713,0.729762,353,750,8702,859,10664,1212
10,0.55,0.222910,0.475248,0.303477,0.29713,0.729762,576,2008,7444,636,10664,1212
4,0.25,0.191955,0.708746,0.302092,0.29713,0.729762,859,3616,5836,353,10664,1212
15,0.80,0.285820,0.319307,0.301637,0.29713,0.729762,387,967,8485,825,10664,1212
3,0.20,0.186962,0.752475,0.299507,0.29713,0.729762,912,3966,5486,300,10664,1212


In [ ]:
# Phase G6: Train the final XGBoost + SMOTE model on all dev data and evaluate on held-out MSL
# Apply SMOTE only to the dev set, retrain once, and evaluate once on untouched test channels
smote_final = SMOTE(random_state=RANDOM_STATE)
X_dev_smote_xgb, y_dev_smote_xgb = smote_final.fit_resample(X_dev, y_dev)

final_scale_pos_weight_smote = (y_dev_smote_xgb == 0).sum() / (y_dev_smote_xgb == 1).sum()

xgb_smote_final = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=final_scale_pos_weight_smote,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
)

xgb_smote_final.fit(X_dev_smote_xgb, y_dev_smote_xgb)
xgb_smote_test_scores = xgb_smote_final.predict_proba(X_test)[:, 1]

xgb_smote_test_metrics = compute_metrics(
    y_true=y_test,
    y_score=xgb_smote_test_scores,
    threshold=xgb_smote_best_row["threshold"],
)

print("XGBoost + SMOTE held-out test metrics:")
print(xgb_smote_test_metrics)

XGBoost + SMOTE held-out test metrics:
{'threshold': np.float64(0.4), 'precision': 0.23443037974683545, 'recall': 0.8638059701492538, 'f1': 0.36877737953006773, 'pr_auc': 0.22616147596936814, 'roc_auc': 0.7068539433090356, 'tp': np.int64(463), 'fp': np.int64(1512), 'tn': np.int64(1889), 'fn': np.int64(73), 'n_rows': 3937, 'n_anomalous': 536}


In [ ]:
# Phase G7: Compare XGBoost baseline vs XGBoost + SMOTE
xgb_compare_df = pd.DataFrame([
    {
        "model": "XGBoost baseline",
        "dev_best_threshold": xgb_best_row["threshold"],
        "test_precision": xgb_test_metrics["precision"],
        "test_recall": xgb_test_metrics["recall"],
        "test_f1": xgb_test_metrics["f1"],
        "test_pr_auc": xgb_test_metrics["pr_auc"],
        "test_roc_auc": xgb_test_metrics["roc_auc"],
        "test_tp": xgb_test_metrics["tp"],
        "test_fp": xgb_test_metrics["fp"],
        "test_tn": xgb_test_metrics["tn"],
        "test_fn": xgb_test_metrics["fn"],
    },
    {
        "model": "XGBoost + SMOTE",
        "dev_best_threshold": xgb_smote_best_row["threshold"],
        "test_precision": xgb_smote_test_metrics["precision"],
        "test_recall": xgb_smote_test_metrics["recall"],
        "test_f1": xgb_smote_test_metrics["f1"],
        "test_pr_auc": xgb_smote_test_metrics["pr_auc"],
        "test_roc_auc": xgb_smote_test_metrics["roc_auc"],
        "test_tp": xgb_smote_test_metrics["tp"],
        "test_fp": xgb_smote_test_metrics["fp"],
        "test_tn": xgb_smote_test_metrics["tn"],
        "test_fn": xgb_smote_test_metrics["fn"],
    },
])

xgb_compare_df

,model,dev_best_threshold,test_precision,test_recall,test_f1,test_pr_auc,test_roc_auc,test_tp,test_fp,test_tn,test_fn
0,XGBoost baseline,0.3,0.210497,0.703358,0.324022,0.252938,0.709510,377,1414,1987,159
1,XGBoost + SMOTE,0.4,0.234430,0.863806,0.368777,0.226161,0.706854,463,1512,1889,73


In [ ]:
# Phase G8: Compare all MSL supervised experiments together
# Combine all MSL supervised results into one table so baseline and SMOTE variants can be compared directly
# This is the final summary used to judge whether SMOTE improved either model
all_supervised_compare_df = pd.DataFrame([
    {
        "model": "Random Forest baseline",
        "dev_best_threshold": rf_best_row["threshold"],
        "test_precision": rf_test_metrics["precision"],
        "test_recall": rf_test_metrics["recall"],
        "test_f1": rf_test_metrics["f1"],
        "test_pr_auc": rf_test_metrics["pr_auc"],
        "test_roc_auc": rf_test_metrics["roc_auc"],
    },
    {
        "model": "Random Forest + SMOTE",
        "dev_best_threshold": rf_smote_best_row["threshold"],
        "test_precision": rf_smote_test_metrics["precision"],
        "test_recall": rf_smote_test_metrics["recall"],
        "test_f1": rf_smote_test_metrics["f1"],
        "test_pr_auc": rf_smote_test_metrics["pr_auc"],
        "test_roc_auc": rf_smote_test_metrics["roc_auc"],
    },
    {
        "model": "XGBoost baseline",
        "dev_best_threshold": xgb_best_row["threshold"],
        "test_precision": xgb_test_metrics["precision"],
        "test_recall": xgb_test_metrics["recall"],
        "test_f1": xgb_test_metrics["f1"],
        "test_pr_auc": xgb_test_metrics["pr_auc"],
        "test_roc_auc": xgb_test_metrics["roc_auc"],
    },
    {
        "model": "XGBoost + SMOTE",
        "dev_best_threshold": xgb_smote_best_row["threshold"],
        "test_precision": xgb_smote_test_metrics["precision"],
        "test_recall": xgb_smote_test_metrics["recall"],
        "test_f1": xgb_smote_test_metrics["f1"],
        "test_pr_auc": xgb_smote_test_metrics["pr_auc"],
        "test_roc_auc": xgb_smote_test_metrics["roc_auc"],
    },
])

all_supervised_compare_df.sort_values("test_f1", ascending=False)

,model,dev_best_threshold,test_precision,test_recall,test_f1,test_pr_auc,test_roc_auc
3,XGBoost + SMOTE,0.4,0.234430,0.863806,0.368777,0.226161,0.706854
0,Random Forest baseline,0.4,0.279749,0.500000,0.358768,0.328580,0.742259
2,XGBoost baseline,0.3,0.210497,0.703358,0.324022,0.252938,0.709510
1,Random Forest + SMOTE,0.6,0.206576,0.621269,0.310056,0.324440,0.732875
